# BreakHis Feature Extraction

## Objective

This notebook extracts deep image embeddings from the trained ResNet18 baseline model developed in the previous stage.

The extracted feature vectors will be used as the image modality input for subsequent multimodal fusion experiments with the clinical model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

print("Feature extraction imports loaded.")

Feature extraction imports loaded.


In [2]:
base_project_path = Path("/Users/sergeysotskiy/Documents/UNI/year 3/Dissertation/dissertation_project")

outputs_path = base_project_path / "outputs"
figures_path = outputs_path / "figures"
metrics_path = outputs_path / "metrics"
reports_path = outputs_path / "reports"
models_path = base_project_path / "models"

figures_path.mkdir(parents=True, exist_ok=True)
metrics_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

print("Reports path:", reports_path)
print("Models path:", models_path)

Reports path: /Users/sergeysotskiy/Documents/UNI/year 3/Dissertation/dissertation_project/outputs/reports
Models path: /Users/sergeysotskiy/Documents/UNI/year 3/Dissertation/dissertation_project/models


In [3]:
train_df = pd.read_csv(reports_path / "train_split.csv")
val_df = pd.read_csv(reports_path / "val_split.csv")
test_df = pd.read_csv(reports_path / "test_split.csv")

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 5536
Validation size: 1186
Test size: 1187


## Preprocessing Setup

The same image size and ImageNet normalization settings used during ResNet18 training are reused here to ensure that extracted image features are consistent with the trained baseline model.

In [4]:
image_size = (224, 224)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

feature_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print("Feature extraction transform ready.")

Feature extraction transform ready.


In [5]:
class BreakHisFeatureDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {"benign": 0, "malignant": 1}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        with Image.open(row["filepath"]) as img:
            image = img.convert("RGB")

        label = self.label_map[row["label"]]
        filepath = row["filepath"]

        if self.transform:
            image = self.transform(image)

        return image, label, filepath

In [6]:
train_dataset = BreakHisFeatureDataset(train_df, transform=feature_transform)
val_dataset = BreakHisFeatureDataset(val_df, transform=feature_transform)
test_dataset = BreakHisFeatureDataset(test_df, transform=feature_transform)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5536
Validation dataset: 1186
Test dataset: 1187


In [7]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 173
Validation batches: 38
Test batches: 38


In [8]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [9]:
model = models.resnet18(weights=None)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model.load_state_dict(torch.load(models_path / "breakhis_resnet18_baseline.pth", map_location=device))
model = model.to(device)
model.eval()

print("Model loaded.")
print("Model parameter device:", next(model.parameters()).device)

Model loaded.
Model parameter device: mps:0


In [10]:
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print("Feature extractor ready.")

Feature extractor ready.


In [11]:
images, labels, filepaths = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    features = feature_extractor(images)

print("Raw feature shape:", features.shape)

Raw feature shape: torch.Size([32, 512, 1, 1])


In [12]:
features = features.view(features.size(0), -1)
print("Flattened feature shape:", features.shape)

Flattened feature shape: torch.Size([32, 512])


## Feature Extraction Function

A reusable function is defined to extract 512-dimensional embeddings for each image in the train, validation, and test splits. These embeddings will later serve as the image feature input for multimodal fusion.

In [14]:
def extract_features(dataloader, feature_extractor, device):
    all_features = []
    all_labels = []
    all_filepaths = []

    feature_extractor.eval()

    with torch.no_grad():
        for images, labels, filepaths in dataloader:
            images = images.to(device)

            features = feature_extractor(images)
            features = features.view(features.size(0), -1)

            all_features.append(features.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_filepaths.extend(filepaths)

    all_features = np.vstack(all_features)

    features_df = pd.DataFrame(all_features)
    features_df["label"] = all_labels
    features_df["filepath"] = all_filepaths

    return features_df

In [15]:
train_features_df = extract_features(train_loader, feature_extractor, device)
val_features_df = extract_features(val_loader, feature_extractor, device)
test_features_df = extract_features(test_loader, feature_extractor, device)

print("Train features shape:", train_features_df.shape)
print("Validation features shape:", val_features_df.shape)
print("Test features shape:", test_features_df.shape)

Train features shape: (5536, 514)
Validation features shape: (1186, 514)
Test features shape: (1187, 514)


In [16]:
train_features_df.to_csv(metrics_path / "breakhis_train_features.csv", index=False)
val_features_df.to_csv(metrics_path / "breakhis_val_features.csv", index=False)
test_features_df.to_csv(metrics_path / "breakhis_test_features.csv", index=False)

print("Feature CSVs saved.")

Feature CSVs saved.


In [17]:
train_features_df.head()

,0,1,2,3,4,5,6,7,8,9,...,504,505,506,507,508,509,510,511,label,filepath
0,0.089164,0.689327,0.020938,0.000000,0.000000,0.000000,0.000000,1.672992,1.928428,2.949181,...,0.114790,0.611499,0.940969,0.000000,0.312568,0.316468,0.232934,0.206695,0,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
1,0.705263,0.536157,2.403208,0.846678,0.416212,1.023999,0.709179,0.120250,1.325927,0.181498,...,0.077053,0.124049,0.061182,0.240798,0.004945,0.576979,0.733759,0.030397,1,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
2,2.244054,1.809127,0.744891,0.765799,2.377185,0.043117,0.031322,0.223340,0.159776,0.772543,...,0.384588,0.793485,0.436995,3.635661,0.031849,1.554540,1.333859,0.199327,1,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
3,2.735526,3.054163,0.501213,0.972428,1.687134,0.021930,0.032390,0.000089,0.013020,1.322015,...,0.958623,0.444661,2.095708,2.753314,0.200393,0.436702,0.520071,0.068639,1,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
4,1.161492,1.492841,0.204054,0.065314,1.995413,2.502908,0.328946,0.165377,2.391004,0.355885,...,0.976770,1.225700,0.751445,1.640889,0.104516,0.468310,2.436186,0.001776,1,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...


## Notebook Conclusion

Deep image embeddings were successfully extracted from the trained ResNet18 baseline model for all BreakHis data splits. Each histopathology image is now represented by a 512-dimensional feature vector, which will be used in the next stage to construct and evaluate multimodal fusion models.